In [3]:
import re
import pandas as pd

In [49]:
def extract_gender(biography, name):
    if not biography or pd.isna(biography):
        return "Unknown"
    
    bio_lower = biography.lower()
    
    # check for explicit gender statement first
    if re.search(r'gender:\s*male', bio_lower):
        return "Male"
    if re.search(r'gender:\s*female', bio_lower):
        return "Female"
    
    # otherwise, count pronouns
    male_pronouns = len(re.findall(r'\b(he|him|his)\b', bio_lower))
    female_pronouns = len(re.findall(r'\b(she|her|hers)\b', bio_lower))
    
    if male_pronouns > female_pronouns:
        return "Male"
    elif female_pronouns > male_pronouns:
        return "Female"
    
    return "Unknown"

In [65]:
# Test it
results_df = pd.read_csv("character_bios_final.csv")

results_df["gender"] = results_df.apply(
    lambda row: extract_gender(row["biography"], row["name"]), axis=1
)

In [53]:
!pip install nltk

In [66]:
results_df.head(5)

,character_id,mal_id,name,biography,gender
0,7,672.0,Gintoki Sakata,"Age: 27, 29 (2 years arc)\nBirthday: October 1...",Male
1,10,286647.0,Kagura,NaN,Unknown
2,12,673.0,Shinpachi Shimura,Birthdate: August 12\nZodiac: Leo\nHeight: 166...,Male
3,14,1533.0,Kotarou Katsura,Age: 20s\nBirthday: June 26 (Cancer)\nHeight: ...,Male
4,16,2650.0,Toushirou Hijikata,Species: Human\nGender: Male\nBirthday: May 5 ...,Male


In [54]:
# use nltk to extract adjectives from characters' bios to get list of personality traits
import nltk
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk import pos_tag, word_tokenize
from collections import Counter

def extract_adjectives(biography):
    if not biography or pd.isna(biography):
        return []
    tokens = word_tokenize(biography)
    tagged = pos_tag(tokens)
    adjectives = [word.lower() for word, tag in tagged 
                  if tag in ("JJ", "JJR", "JJS")
                  and len(word) > 2]
    return list(set(adjectives))

# Test on a few first
for _, row in results_df.head(10).iterrows():
    adjs = extract_adjectives(row["biography"])
    print(f"{row['name']}: {adjs}\n")

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/marianama/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to /Users/marianama/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/marianama/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Gintoki Sakata: ['most', 'latest', 'such', 'reckless', 'impressive', 'oppressive', 'dentist', 'other', 'old', 'second', 'fearsome', 'odd', 'several', 'great', 'unmatched', 'pachinko', 'jump', 'sweet', 'only', 'big', 'similar', 'initial', 'due', 'major', 'white', 'famous', 'female']

Kagura: []

Shinpachi Shimura: ['competent', 'comedian', '21-22', 'meek', 'leo', 'spilled']

Kotarou Katsura: ['japanese', 'such', 'first', 'left', 'violent', 'wanted', 'other', 'historical', 'important', 'odd', 'outdated', 'dramatic', 'skull', 'past', 'right', 'uncanny', 'similar', 'initial', 'metallic', 'due', 'subsequent', 'former', 'same', 'terrorist', 'complete', 'english']

Toushirou Hijikata: ['right-handed', 'such', 'rival', 'hot-tempered', 'taurus', 'cold', 'famed', 'trouble-maker', 'demonic', 'close', 'popular', 'loyal', 'fearsome', 'constant', 'emotional', 'male', 'compassionate', 'big', 'many', 'human', 'disciplinary', 'good']

Sougo Okita: ['sakura-blossom', 'japanese', 'original', 'unfinished'

In [55]:
from collections import Counter

# extract adjectives from all bios
results_df["all_adjectives"] = results_df["biography"].apply(extract_adjectives)

# get most common across entire dataset
all_adj = []
for adjs in results_df["all_adjectives"].dropna():
    all_adj.extend(adjs)

top_adj = Counter(all_adj).most_common(500)
for adj, count in top_adj:
    print(f"{adj}: {count}")

KeyboardInterrupt: 

In [56]:
# personality adjectives from the list
personality_adjectives = {
    # stereotypically "positive" traits
    "friendly", "cheerful", "kind-hearted", "gentle", "calm", "loyal", "polite", "nice", "kind",
    "honest", "confident", "energetic", "optimistic", "playful", "carefree",
    "innocent", "humble", "reliable", "responsible", "caring", "supportive",
    "helpful", "determined", "passionate", "enthusiastic", "outgoing", "charismatic",
    "mature", "independent", "perceptive", "observant", "thoughtful", "logical",
    "intelligent", "smart", "bright", "capable", "talented", "athletic", "skilled",
    "cool", "elegant", "noble", "protective", "peaceful", "powerful", "pure", "good-natured",
    "popular", "beautiful", "sweet", "sharp", "innocent", "attractive", "talented", "playful",
    "level-headed", "soft-spoken", "strong-willed", "strong", "laid-back", "easygoing", "mature", 
    "successful", "charismatic", "curious", "trusted", "knowledgeable",
    
    # stereotypically "negative" traits
    "arrogant", "selfish", "shy", "timid", "lazy", "naive", "clumsy", "competitive",
    "stubborn", "aggressive", "violent", "cruel", "ruthless", "manipulative",
    "sadistic", "jealous", "childish", "reckless", "impulsive", "perverted",
    "eccentric", "weird", "obsessive", "cynical", "sarcastic", "aloof", 
    "stoic", "serious", "strict", "harsh", "rude", "rebellious", "competitive",
    "proud", "cowardly", "nervous", "awkward", "sensitive", "emotional", "dramatic",
    "mischievous", "wild", "crazy", "mad", "hostile", "fierce", "brutal",
    "mysterious", "secretive", "reserved", "introverted", "tomboyish", "timid",
    "short-tempered", "tsundere", "oblivious", "cautious", "suspicious", "sensitive",
    "reluctant", "loud", "talkative", "bold", "blunt", "direct", "weak", "angry", "oblivious"
}

In [57]:
# Function to extract specified traits from each character's biography
def extract_personality_traits(biography):
    if not biography or pd.isna(biography):
        return []
    
    bio_lower = biography.lower()
    return [trait for trait in personality_adjectives 
            if re.search(rf'\b{trait}\b', bio_lower)]

# Apply to dataset
results_df["personality_traits"] = results_df["biography"].apply(extract_personality_traits)

# Check results
for _, row in results_df.head(10).iterrows():
    print(f"{row['name']}: {row['personality_traits']}")

# See how many characters have at least one trait
has_traits = results_df["personality_traits"].apply(len) > 0
print(f"\nCharacters with at least one trait: {has_traits.sum()} ({has_traits.mean():.1%})")

Gintoki Sakata: ['reckless', 'sweet']
Kagura: []
Shinpachi Shimura: []
Kotarou Katsura: ['violent', 'dramatic']
Toushirou Hijikata: ['loyal', 'emotional', 'popular']
Sougo Okita: ['childish']
Shinsuke Takasugi: ['skilled']
Taizou Hasegawa: ['successful']
Tsukuyo: ['ruthless']
Kamui: []

Characters with at least one trait: 4789 (37.5%)


In [59]:
# Test on specific character to see output
bio = results_df[results_df["mal_id"] == 155904.0]["biography"].values[0]
print(bio)
print(extract_personality_traits(bio))

Age: 17
Birth place: Rokushoukan
Hair color: Black (WN), Dark Green (LN)
Eye color: (Black (WN), Purple (LN)
Height: 153 cm (5'0")

Maomao became a maidservant to the Emperor's harem after being kidnapped and sold. She used to live with her father as a pharmacist in the red light district and is very knowledgeable about medicine and poisons but isn't very interested in other humans.

She is skinny and short, and her plain face is covered with freckles.

AbilitiesMaomao's name is derived from 猫 (māo), the Chinese word for cat. This reinforces the fact that she is often described as a cat or cat-like.


(Source: The Apothecary Diaries Wiki, edited)
['knowledgeable']


In [60]:
# Now check bias / ratio of female to male personalitys extracted
results_df["has_traits"] = results_df["personality_traits"].apply(len) > 0

print("Trait extraction rate by gender:")
print(results_df.groupby("gender")["has_traits"].mean())

print("\nGender distribution overall:")
print(results_df["gender"].value_counts())

Trait extraction rate by gender:
gender
Female     0.558942
Male       0.549083
Unknown    0.046089
Name: has_traits, dtype: float64

Gender distribution overall:
gender
Unknown    4513
Female     4233
Male       4034
Name: count, dtype: int64


In [62]:
print(results_df[results_df["name"] == "Tooru Honda"]["personality_traits"].values[0])

['optimistic', 'polite', 'kind']


In [63]:
# Save results to csv
results_df.to_csv("character_personalities_final.csv", index=False)

In [64]:
results_df.sample(50)

,character_id,mal_id,name,biography,gender,personality_traits,has_traits
7741,20791,215857.0,Katori-kun,Occupation: Sales department\nSpecies: Human-P...,Male,[cool],True
5421,8872,110899.0,Ikuya Kirishima,Ikuya Kirishima was the first-year junior high...,Male,[],False
4961,12152,11771.0,Arslan,"Age: 11, 14 (post time-skip)\nBirthday: May 23...",Male,"[intelligent, charismatic]",True
581,1463,111245.0,Rudeus Greyrat,Rudeus Greyrat is a reincarnated NEET who died...,Male,[],False
11802,42447,142394.0,Niwatori,NaN,Unknown,[],False
9485,27871,128504.0,Mugi Awaya,Another high school student who is in love wit...,Unknown,[],False
2331,5238,172491.0,Midori Asakusa,"Asakusa is a freshman at Shibahama Highschool,...",Female,[],False
5115,12511,274448.0,Hosaka,NaN,Unknown,[],False
8940,25586,90169.0,Minori Koganuma,A well-endowed member of the JSDF assigned to ...,Unknown,[skilled],True
7445,19573,151107.0,Nagahide Niwa,NaN,Unknown,[],False


In [67]:
import pandas as pd
import ast
from collections import Counter

chars = pd.read_csv("../cleaned_data/character_personalities_final.csv")

def parse_traits(x):
    if pd.isna(x) or x == "[]" or x == "":
        return []
    try:
        return ast.literal_eval(x)
    except:
        return []

chars["traits_list"] = chars["personality_traits"].apply(parse_traits)

valid = chars[chars["gender"].isin(["Male", "Female"])].copy()
male_total = (valid["gender"] == "Male").sum()
female_total = (valid["gender"] == "Female").sum()

male_counts = Counter()
female_counts = Counter()

for _, row in valid.iterrows():
    if row["gender"] == "Male":
        male_counts.update(row["traits_list"])
    else:
        female_counts.update(row["traits_list"])

all_traits = set(male_counts.keys()) | set(female_counts.keys())

rows = []
for trait in all_traits:
    m = male_counts[trait]
    f = female_counts[trait]
    total = m + f
    rows.append({
        "trait": trait,
        "male_count": m,
        "female_count": f,
        "total": total,
        "male_pct": round(m / male_total * 100, 1),
        "female_pct": round(f / female_total * 100, 1),
        "female_share": round(f / total * 100, 1) if total > 0 else 0
    })

trait_df = pd.DataFrame(rows).sort_values("total", ascending=False)

# Display with all columns visible
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 10)
pd.set_option("display.width", 120)

print(trait_df[trait_df["total"] >= 20].to_string(index=False))

         trait  male_count  female_count  total  male_pct  female_pct  female_share
        strong         261           242    503       6.5         5.7          48.1
          kind         213           246    459       5.3         5.8          53.6
      powerful         185           111    296       4.6         2.6          37.5
       skilled         149            99    248       3.7         2.3          39.9
       popular         116           125    241       2.9         3.0          51.9
       serious         149            92    241       3.7         2.2          38.2
          calm         145            87    232       3.6         2.1          37.5
     beautiful          60           169    229       1.5         4.0          73.8
           shy          59           168    227       1.5         4.0          74.0
       capable         104           103    207       2.6         2.4          49.8
      friendly         100           104    204       2.5         2.5       

In [69]:
trait_df = pd.DataFrame(rows).sort_values("female_share", ascending=False)

# Display with all columns visible
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 10)
pd.set_option("display.width", 120)

print(trait_df[trait_df["total"] >= 20].to_string(index=False))

         trait  male_count  female_count  total  male_pct  female_pct  female_share
     tomboyish           3            29     32       0.1         0.7          90.6
      tsundere           7            53     60       0.2         1.3          88.3
        clumsy          15            61     76       0.4         1.4          80.3
       elegant           8            28     36       0.2         0.7          77.8
   independent          15            44     59       0.4         1.0          74.6
           shy          59           168    227       1.5         4.0          74.0
     beautiful          60           169    229       1.5         4.0          73.8
   soft-spoken           8            22     30       0.2         0.5          73.3
         timid          19            50     69       0.5         1.2          72.5
     energetic          33            83    116       0.8         2.0          71.6
        mature          35            84    119       0.9         2.0       

In [70]:
# Score = skew * log(count) — rewards both extremes and volume
import numpy as np

trait_df["male_skew"] = 100 - trait_df["female_share"]  # higher = more male
trait_df["female_skew"] = trait_df["female_share"]       # higher = more female

trait_df["male_score"] = trait_df["male_skew"] * np.log(trait_df["total"])
trait_df["female_score"] = trait_df["female_skew"] * np.log(trait_df["total"])

print("Top male-skewed (skew + volume):")
print(trait_df.nlargest(10, "male_score")[["trait", "total", "female_share", "male_score"]].to_string(index=False))

print("\nTop female-skewed (skew + volume):")
print(trait_df.nlargest(10, "female_score")[["trait", "total", "female_share", "female_score"]].to_string(index=False))

Top male-skewed (skew + volume):
   trait  total  female_share  male_score
powerful    296          37.5  355.647466
   loyal    149          30.9  345.772690
    calm    232          37.5  340.421086
 serious    241          38.2  338.960450
 skilled    248          39.9  331.357068
  strong    503          48.1  322.848630
arrogant     93          33.3  302.324386
    cool    124          39.5  291.627035
    kind    459          53.6  284.387930
    weak    152          43.4  284.351637

Top female-skewed (skew + volume):
    trait  total  female_share  female_score
      shy    227          74.0    401.446301
beautiful    229          73.8    401.008684
 tsundere     60          88.3    361.530625
 cheerful    179          68.7    356.373405
   clumsy     76          80.3    347.757887
energetic    116          71.6    340.357058
   mature    119          70.6    337.406119
    sweet    126          69.0    333.703452
     kind    459          53.6    328.517091
tomboyish     32   